# Physical Dissector — Demo

Simulates a 200 x 200 x 4 unit volume divided into 4 image layers of thickness 1 (z-axis).  
Dummy cells are cylinders in x-y that span either 1 or 2 z-layers (to mimic full cells vs cells split by sectioning/imaging).  
z = 0 is the **top** of the volume. Counting proceeds top-down.

**Counting rule:** each cylinder is counted once, in the first layer where it appears.  
That is the layer containing `top_z` (the top face of the cylinder).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import warnings
warnings.filterwarnings('ignore')

## 1. Parameters

In [ ]:
# Volume dimensions
VOL_X = 200
VOL_Y = 200
VOL_Z = 4  # 4 image layers

# Slicing
SLICE_THICKNESS = 1
N_SLICES = VOL_Z // SLICE_THICKNESS  # 4

# Dummy cell geometry (cylinders)
FOOTPRINT_RADIUS = 8
SPAN_OPTIONS = [1, 2]  # cylinders span 1 or 2 layers

# Reproducibility
SEED = 42
N_CELLS = 30

# Slice colours for plotting
SLICE_COLORS = ['#4f8ef7', '#2ecfa3', '#f7b84f', '#e05c7a']

## 2. Generate dummy cells

In [ ]:
rng = np.random.default_rng(SEED)

def generate_cells(n, rng, span_options=SPAN_OPTIONS):
    """
    Generate n dummy cells as vertical cylinders.
    Each cylinder has a fixed x-y footprint radius and spans either 1 or 2 z-layers.
    """
    records = []
    for _ in range(n):
        cx = rng.uniform(0, VOL_X)
        cy = rng.uniform(0, VOL_Y)

        span_layers = int(rng.choice(span_options))
        max_start_layer = N_SLICES - span_layers
        start_layer = int(rng.integers(0, max_start_layer + 1))
        end_layer = start_layer + span_layers - 1

        top_z = start_layer * SLICE_THICKNESS
        bot_z = (end_layer + 1) * SLICE_THICKNESS

        records.append({
            'cx': cx, 'cy': cy,
            'top_z': top_z,
            'bot_z': bot_z,
            'span_layers': span_layers,
            'start_layer': start_layer,
            'end_layer': end_layer,
        })

    return pd.DataFrame(records).reset_index(drop=True)

cells = generate_cells(N_CELLS, rng)
print(f"Generated {len(cells)} cells")
cells.head()

## 3. Dissector counting logic

Each cylinder is assigned to exactly one layer: the first (top-most) layer where it appears.  
With this setup, that is just the cylinder start layer (`top_z / SLICE_THICKNESS`).

In [ ]:
def assign_slice(top_z, slice_thickness=SLICE_THICKNESS, n_slices=N_SLICES):
    """
    Return the index (0-based) of the layer in which this cylinder
    first appears, moving top-down through the volume.
    """
    idx = int(top_z // slice_thickness)
    return min(idx, n_slices - 1)

cells['slice_idx'] = cells['top_z'].apply(assign_slice)
cells['slice_label'] = cells['slice_idx'].apply(
    lambda i: f'slice {i+1} (z {i*SLICE_THICKNESS}-{(i+1)*SLICE_THICKNESS})'
 )

print("Cells per slice:")
print(cells.groupby('slice_label').size().to_string())
print(f"\nTotal counted: {len(cells)}  (should equal number generated: {N_CELLS})")

## 4. Summary table

In [ ]:
display_cols = ['cx','cy','top_z','bot_z','span_layers','start_layer','end_layer','slice_idx','slice_label']
df_display = cells[display_cols].copy()
for col in ['cx','cy','top_z','bot_z']:
    df_display[col] = df_display[col].round(2)
df_display

## 5. 3D visualisation

In [ ]:
def draw_slice_box(ax, z_start, z_end, color, alpha=0.07):
    """Draw a transparent box representing one slice."""
    x0, x1 = 0, VOL_X
    y0, y1 = 0, VOL_Y
    faces = [
        [[x0,y0,z_start],[x1,y0,z_start],[x1,y1,z_start],[x0,y1,z_start]],
        [[x0,y0,z_end  ],[x1,y0,z_end  ],[x1,y1,z_end  ],[x0,y1,z_end  ]],
        [[x0,y0,z_start],[x1,y0,z_start],[x1,y0,z_end  ],[x0,y0,z_end  ]],
        [[x0,y1,z_start],[x1,y1,z_start],[x1,y1,z_end  ],[x0,y1,z_end  ]],
        [[x0,y0,z_start],[x0,y1,z_start],[x0,y1,z_end  ],[x0,y0,z_end  ]],
        [[x1,y0,z_start],[x1,y1,z_start],[x1,y1,z_end  ],[x1,y0,z_end  ]],
    ]
    poly = Poly3DCollection(faces, alpha=alpha, facecolor=color, edgecolor=color, linewidth=0.3)
    ax.add_collection3d(poly)


def draw_cylinder(ax, cx, cy, z0, z1, radius, color, alpha=0.6):
    """Draw a vertical cylinder from z0 to z1."""
    theta = np.linspace(0, 2 * np.pi, 30)
    z = np.linspace(z0, z1, 2)
    theta_grid, z_grid = np.meshgrid(theta, z)
    x_grid = cx + radius * np.cos(theta_grid)
    y_grid = cy + radius * np.sin(theta_grid)
    ax.plot_surface(x_grid, y_grid, z_grid, color=color, alpha=alpha, linewidth=0)


fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Draw slice boxes
for i in range(N_SLICES):
    draw_slice_box(ax, i*SLICE_THICKNESS, (i+1)*SLICE_THICKNESS, SLICE_COLORS[i], alpha=0.06)

# Draw cells coloured by assigned slice
for _, row in cells.iterrows():
    color = SLICE_COLORS[row['slice_idx']]
    draw_cylinder(ax, row['cx'], row['cy'], row['top_z'], row['bot_z'], FOOTPRINT_RADIUS, color, alpha=0.55)
    # Mark first-appearance layer with a small dot
    ax.scatter(row['cx'], row['cy'], row['top_z'], color=color, s=18, zorder=5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z (top=0)')
ax.set_xlim(0, VOL_X)
ax.set_ylim(0, VOL_Y)
ax.set_zlim(0, VOL_Z)
ax.invert_zaxis()  # z=0 at top
ax.set_title('Physical dissector — first appearance counting\n(dots mark first-appearance layer per cell)', fontsize=11)

legend_patches = [
    mpatches.Patch(color=SLICE_COLORS[i], label=f'Slice {i+1} (z {i*SLICE_THICKNESS}-{(i+1)*SLICE_THICKNESS})')
    for i in range(N_SLICES)
 ]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Per-slice count bar chart

In [ ]:
counts = cells.groupby('slice_idx').size().reindex(range(N_SLICES), fill_value=0)
labels = [f'Slice {i+1}\nz {i*SLICE_THICKNESS}-{(i+1)*SLICE_THICKNESS}' for i in range(N_SLICES)]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, counts.values, color=SLICE_COLORS, edgecolor='none', width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, str(val),
            ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Cells counted (first appearance)')
ax.set_title(f'Dissector counts per slice  —  total = {counts.sum()}')
ax.set_ylim(0, counts.max() + 3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Total cells in volume:  {len(cells)}")
print(f"1-layer cells: {(cells['span_layers'] == 1).sum()}")
print(f"2-layer cells: {(cells['span_layers'] == 2).sum()}")
print(f"Sum of per-slice counts: {counts.sum()}  <- must equal total")

## 7. Cross-section views — one per slice

Shows which cylinders are visible in each 1-unit z layer (any cylinder intersecting that z range),
and which ones are being **counted** in that slice (first appearance = white centre dot).

In [ ]:
fig, axes = plt.subplots(1, N_SLICES, figsize=(14, 4), sharex=True, sharey=True)

for i, ax in enumerate(axes):
    z0 = i * SLICE_THICKNESS
    z1 = (i + 1) * SLICE_THICKNESS
    color = SLICE_COLORS[i]

    # All cylinders that intersect this slice
    visible = cells[(cells['top_z'] < z1) & (cells['bot_z'] > z0)]

    ax.set_facecolor('#0a0a12')
    ax.set_xlim(0, VOL_X)
    ax.set_ylim(0, VOL_Y)
    ax.set_aspect('equal')
    ax.set_title(f'Slice {i+1}  z {z0}-{z1}', fontsize=10, color=color)
    ax.set_xlabel('X')
    if i == 0:
        ax.set_ylabel('Y')

    for _, row in visible.iterrows():
        is_counted = (row['slice_idx'] == i)
        circle = plt.Circle(
            (row['cx'], row['cy']),
            FOOTPRINT_RADIUS,
            color=color,
            alpha=0.8 if is_counted else 0.2,
            fill=True,
            linewidth=0
        )
        ax.add_patch(circle)
        if is_counted:
            ax.plot(row['cx'], row['cy'], 'w.', markersize=4)

    counted_n = (cells['slice_idx'] == i).sum()
    ax.text(4, VOL_Y - 8, f'counted: {counted_n}', color=color, fontsize=9)
    ax.text(4, VOL_Y - 18, f'visible: {len(visible)}', color='#888', fontsize=9)

plt.suptitle('Cross-section per layer  —  bright = counted (first appearance), dim = visible but not counted', fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

## 8. Verify no double-counting

Every cell must appear in exactly one slice.

In [ ]:
assert cells['slice_idx'].notna().all(), "Some cells have no assigned slice"
assert len(cells) == counts.sum(), "Count mismatch — double counting or missed cells"
assert cells['slice_idx'].between(0, N_SLICES-1).all(), "Slice index out of range"

print("All assertions passed.")
print(f"  {len(cells)} cells generated, {counts.sum()} counted across {N_SLICES} slices.")
print(f"  No double-counting.")

---
## Notes for extension to real data

When moving from dummy cylinders to real WSI-derived cell labels:

1. **Replace `generate_cells()`** with a loader that reads centroid coordinates from your segmentation output (e.g. a CSV or GeoJSON from QuPath/Cellpose).
2. **Registration tolerance**: if WSI registration has residual error, replace the hard `top_z` threshold with a neighbourhood search - check for a matching centroid within `search_radius` pixels in the look-up section before declaring a cell absent.
3. **Variable footprint / z-span**: use per-cell x-y footprint radius and inferred z-span from consecutive stack planes instead of fixed dummy values.
4. **Guard zones**: exclude cells whose `top_z` falls within `GUARD` units of z=0 or z=VOL_Z to avoid surface artefacts. Typically 2-5 um each end.
5. **Fractionator equation** (total N estimate):  
   `N_total = (1/ssf) * (1/asf) * (1/tsf) * sum(Q-)`  
   where ssf = section sampling fraction, asf = area sampling fraction, tsf = thickness sampling fraction, Q- = counted cells.
6. **Coefficient of error**: use the Gundersen CE formula on the per-slice counts to assess sampling efficiency.